In [2]:
import pandas as pd

In [3]:
df_autoarm = pd.read_csv("./data/raw/autoarm_items.csv")
df_autofrei = pd.read_csv("./data/raw/autofrei_items.csv")
df_entities = pd.read_csv("./data/metadata/all_entities.csv")

In [5]:
#meetings sind einzelne Sitzungen
#proposals sind Vorgänge, denen mehrere Sitzungen zugeordnet sein können.
#Achtung: Nicht alle Kommunen ordnen ihre Sitzungen Proposals zu
print("Anzahl gematchter Chunks pro Gruppentyp:")
df_autofrei["groupType"].value_counts()


Anzahl gematchter Chunks pro Gruppentyp:


groupType
meeting     8019
proposal    4698
Name: count, dtype: int64

In [7]:
def get_unique_meetings(df, groupType = "meeting"):
    """gruppiert die Chunks, die zum selben Element gehören"""
    
    result = pd.DataFrame(columns=["groupKey", "meeting_date", "chunkCount", "proposalId", "entityId", "entityLevel"])

    rows = df[df["groupType"] == groupType]
    unique_groupKeys = rows["groupKey"].drop_duplicates()

    for groupKey in unique_groupKeys:
        chunk_count = len(df[df["groupKey"] == groupKey])
        first_instance = df[df["groupKey"] == groupKey].iloc[0]

        result.loc[len(result)] = {
            "groupKey": groupKey,
            "date": first_instance["date"],
            "chunkCount": chunk_count,
            "proposalId": first_instance["proposalId"],
            "entityId": first_instance["entityId"],
            "entityLevel": first_instance["entityLevel"]
        }

    return result

In [14]:
meetings = get_unique_meetings(df_autofrei, groupType="meeting")
proposals = get_unique_meetings(df_autofrei, groupType="proposal")

In [12]:
def add_entityName(data, entities):
    data["entityName"] = data["entityId"].map(entities.drop_duplicates(subset=["id"]).set_index("id")["name"])
    return data


In [13]:
proposals = add_entityName(proposals, df_entities)

KeyError: 'entityId'

In [11]:
proposals

0                                      Leipzig, Stadt
1                                         Bonn, Stadt
2                                 Herford, Hansestadt
3                                      Boxberg, Stadt
4                           Menden (Sauerland), Stadt
                            ...                      
2296    Homberg (Efze), Reformationsstadt, Kreisstadt
2297                                  Dortmund, Stadt
2298                                 Ettlingen, Stadt
2299                                    Bochum, Stadt
2300                                  Alb-Donau-Kreis
Name: entityId, Length: 2301, dtype: str

In [19]:
meetings["entityName"] = meetings["entityId"].map(df_entities.drop_duplicates(subset=["id"]).set_index("id")["name"])


In [30]:
proposals

0                                      Leipzig, Stadt
1                                         Bonn, Stadt
2                                 Herford, Hansestadt
3                                      Boxberg, Stadt
4                           Menden (Sauerland), Stadt
                            ...                      
2296    Homberg (Efze), Reformationsstadt, Kreisstadt
2297                                  Dortmund, Stadt
2298                                 Ettlingen, Stadt
2299                                    Bochum, Stadt
2300                                  Alb-Donau-Kreis
Name: entityId, Length: 2301, dtype: str

In [22]:
proposals["entityName"].value_counts()

KeyError: 'entityName'